# Shopping Behaviour Analysis
### What Influences Consumer Shopping Behaviour?
**Student:** Oluwatoyin Oluwafunmilayo Bakare | **ID:** 22585411
---


## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import datetime
import warnings
warnings.filterwarnings('ignore')
from statsmodels.graphics.mosaicplot import mosaic
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
import shap

sns.set(
    {"figure.figsize": (8, 6)},
    style='ticks',
    color_codes=True,
    font_scale=0.8
)
%config InlineBackend.figure_format = 'retina'
PALETTE = 'Set2'
print("All libraries imported successfully.")


## 2. Data Loading and Initial Exploration

In [ ]:
df = pd.read_csv('shopping_behavior_updated.csv')
print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())


In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
print("Missing Values:")
print(df.isnull().sum())


In [ ]:
df.describe(include='all')

## 3. Exploratory Data Analysis (EDA)
### 3.1 Distribution of Key Numeric Variables


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df['Age'], bins=20, color='steelblue', ax=axes[0], kde=True)
axes[0].set_title('Age Distribution', fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Count')

sns.histplot(df['Purchase Amount (USD)'], bins=20, color='coral', ax=axes[1], kde=True)
axes[1].set_title('Purchase Amount Distribution', fontweight='bold')
axes[1].set_xlabel('Purchase Amount (USD)')

sns.histplot(df['Review Rating'], bins=15, color='mediumseagreen', ax=axes[2], kde=True)
axes[2].set_title('Review Rating Distribution', fontweight='bold')
axes[2].set_xlabel('Review Rating')
plt.suptitle('Distribution of Key Numeric Variables', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### 3.2 Demographics: Category and Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cat_counts = df['Category'].value_counts()
axes[0].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette(PALETTE, len(cat_counts)), startangle=140)
axes[0].set_title('Purchase Category Breakdown', fontweight='bold')

gen_counts = df['Gender'].value_counts()
axes[1].bar(gen_counts.index, gen_counts.values,
            color=sns.color_palette(PALETTE, 2), edgecolor='white')
axes[1].set_title('Gender Distribution', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(gen_counts.values):
    axes[1].text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.suptitle('Shopping Demographics Overview', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 3.3 Purchase Amount by Category and Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Category', y='Purchase Amount (USD)', palette=PALETTE, ax=axes[0])
axes[0].set_title('Purchase Amount by Category', fontweight='bold')

sns.boxplot(data=df, x='Gender', y='Purchase Amount (USD)', palette=PALETTE, ax=axes[1])
axes[1].set_title('Purchase Amount by Gender', fontweight='bold')
plt.suptitle('Purchase Amount Distribution by Category and Gender', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 3.4 Seasonal Shopping Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
season_avg = df.groupby('Season')['Purchase Amount (USD)'].mean().reindex(['Spring','Summer','Fall','Winter'])
axes[0].bar(season_avg.index, season_avg.values,
            color=sns.color_palette(PALETTE, 4), edgecolor='white')
axes[0].set_title('Average Purchase Amount by Season', fontweight='bold')
axes[0].set_ylabel('Average Amount (USD)')
for i, v in enumerate(season_avg.values):
    axes[0].text(i, v + 0.3, f'${v:.1f}', ha='center', fontsize=9)

season_cat = df.groupby(['Season','Category']).size().unstack(fill_value=0)
season_cat = season_cat.reindex(['Spring','Summer','Fall','Winter'])
season_cat.plot(kind='bar', ax=axes[1], colormap=PALETTE, edgecolor='white', rot=0)
axes[1].set_title('Purchase Count by Season and Category', fontweight='bold')
axes[1].set_xlabel('Season'); axes[1].set_ylabel('Count')
axes[1].legend(title='Category', bbox_to_anchor=(1, 1))
plt.suptitle('Seasonal Shopping Patterns', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 3.5 Impact of Discounts and Promo Codes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
disc_avg = df.groupby('Discount Applied')['Purchase Amount (USD)'].mean()
axes[0].bar(disc_avg.index, disc_avg.values, color=['#DD8452','#4C72B0'], edgecolor='white')
axes[0].set_title('Avg Purchase: Discount Applied', fontweight='bold')
axes[0].set_ylabel('Average Amount (USD)')
for i, v in enumerate(disc_avg.values):
    axes[0].text(i, v + 0.3, f'${v:.1f}', ha='center', fontweight='bold')

promo_avg = df.groupby('Promo Code Used')['Purchase Amount (USD)'].mean()
axes[1].bar(promo_avg.index, promo_avg.values, color=['#55A868','#C44E52'], edgecolor='white')
axes[1].set_title('Avg Purchase: Promo Code Used', fontweight='bold')
axes[1].set_ylabel('Average Amount (USD)')
for i, v in enumerate(promo_avg.values):
    axes[1].text(i, v + 0.3, f'${v:.1f}', ha='center', fontweight='bold')
plt.suptitle('Impact of Discounts and Promo Codes on Spending', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 3.6 Subscription Status and Customer Behaviour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sub_avg = df.groupby('Subscription Status')['Purchase Amount (USD)'].mean()
axes[0].bar(sub_avg.index, sub_avg.values,
            color=sns.color_palette(PALETTE, 2), edgecolor='white')
axes[0].set_title('Avg Purchase by Subscription Status', fontweight='bold')
axes[0].set_ylabel('Average Amount (USD)')
for i, v in enumerate(sub_avg.values):
    axes[0].text(i, v + 0.3, f'${v:.1f}', ha='center', fontweight='bold')

sub_prev = df.groupby('Subscription Status')['Previous Purchases'].mean()
axes[1].bar(sub_prev.index, sub_prev.values,
            color=sns.color_palette(PALETTE, 2), edgecolor='white')
axes[1].set_title('Avg Previous Purchases by Subscription', fontweight='bold')
axes[1].set_ylabel('Avg Previous Purchases')
for i, v in enumerate(sub_prev.values):
    axes[1].text(i, v + 0.1, f'{v:.1f}', ha='center', fontweight='bold')
plt.suptitle('Subscription Status and Customer Behaviour', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 3.7 Payment Method and Shipping Preferences

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pay_counts = df['Payment Method'].value_counts()
sns.barplot(x=pay_counts.values, y=pay_counts.index, palette=PALETTE, ax=axes[0])
axes[0].set_title('Payment Method Frequency', fontweight='bold')
axes[0].set_xlabel('Count')

ship_avg = df.groupby('Shipping Type')['Purchase Amount (USD)'].mean().sort_values(ascending=False)
sns.barplot(x=ship_avg.values, y=ship_avg.index, palette=PALETTE, ax=axes[1])
axes[1].set_title('Avg Purchase by Shipping Type', fontweight='bold')
axes[1].set_xlabel('Average Amount (USD)')
plt.suptitle('Payment Methods and Shipping Preferences', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 3.8 Age vs Purchase Amount

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cat, grp in df.groupby('Category'):
    ax.scatter(grp['Age'], grp['Purchase Amount (USD)'],
               alpha=0.25, s=12, label=cat)
ax.set_title('Age vs Purchase Amount by Category', fontweight='bold', fontsize=13)
ax.set_xlabel('Age'); ax.set_ylabel('Purchase Amount (USD)')
ax.legend(title='Category')
plt.tight_layout(); plt.show()


### 3.9 Frequency of Purchases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
freq_order = ['Weekly','Fortnightly','Bi-Weekly','Monthly','Quarterly','Every 3 Months','Annually']
freq_counts = df['Frequency of Purchases'].value_counts().reindex(freq_order)
sns.barplot(x=freq_counts.values, y=freq_counts.index, palette='Blues_r', ax=axes[0])
axes[0].set_title('Frequency of Purchases Distribution', fontweight='bold')
axes[0].set_xlabel('Count')

freq_amt = df.groupby('Frequency of Purchases')['Purchase Amount (USD)'].mean().reindex(freq_order)
sns.barplot(x=freq_amt.values, y=freq_amt.index, palette='Oranges_r', ax=axes[1])
axes[1].set_title('Avg Purchase Amount by Frequency', fontweight='bold')
axes[1].set_xlabel('Average Amount (USD)')
plt.suptitle('Purchase Frequency Analysis', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 3.10 Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
num_df = df[['Age','Purchase Amount (USD)','Review Rating','Previous Purchases']]
sns.heatmap(num_df.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — Numeric Features', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 3.11 Subscription Status vs Discount Applied

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cross = pd.crosstab(df['Subscription Status'], df['Discount Applied'])
cross.plot(kind='bar', ax=ax, colormap=PALETTE, edgecolor='white', rot=0)
ax.set_title('Subscription Status vs Discount Applied', fontweight='bold', fontsize=13)
ax.set_xlabel('Subscription Status'); ax.set_ylabel('Count')
ax.legend(title='Discount Applied')
plt.tight_layout(); plt.show()


## 4. Data Preprocessing and Feature Engineering
### 4.1 Encoding Categorical Variables


In [ ]:
df_ml = df.drop(columns=['Customer ID']).copy()

# Encode binary categorical columns
binary_cols = ['Gender', 'Subscription Status', 'Discount Applied', 'Promo Code Used']
for col in binary_cols:
    df_ml[col] = LabelEncoder().fit_transform(df_ml[col])

# Encode multi-class categorical columns
cat_cols = ['Item Purchased','Category','Location','Size','Color',
            'Season','Shipping Type','Payment Method','Frequency of Purchases']
le_dict = {}
for col in cat_cols:
    le_dict[col] = LabelEncoder()
    df_ml[col] = le_dict[col].fit_transform(df_ml[col])

print("Encoding complete. Sample:")
df_ml.head()


## 5. Feature Selection
### 5.1 SelectKBest (F-regression, k=8)


In [ ]:
X = df_ml.drop(columns=['Purchase Amount (USD)'])
y = df_ml['Purchase Amount (USD)']

selector = SelectKBest(f_regression, k=8)
selector.fit(X, y)
selected_features = X.columns[selector.get_support()].tolist()
print("Selected features:", selected_features)

# Visualise feature scores
fig, ax = plt.subplots(figsize=(10, 5))
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
sns.barplot(x=scores.values, y=scores.index, palette='viridis', ax=ax)
ax.set_title('Feature Selection Scores (SelectKBest — F-regression)', fontweight='bold')
ax.set_xlabel('F-Score'); ax.set_ylabel('Feature')
plt.tight_layout(); plt.show()


### 5.2 Train-Test Split and Scaling

In [ ]:
X_sel = X[selected_features]
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set: {X_train_sc.shape}")
print(f"Test set:     {X_test_sc.shape}")


## 6. Model Building
### 6.1 Initialise Models


In [ ]:
lr  = LinearRegression()
rfr = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
gbr = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
ensemble = VotingRegressor([('lr', lr), ('rfr', rfr), ('gbr', gbr)])

models = {
    'Linear Regression':  lr,
    'Random Forest':      rfr,
    'Gradient Boosting':  gbr,
    'Ensemble':           ensemble
}


### 6.2 Train and Evaluate All Models

In [ ]:
results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    cv = cross_val_score(model, X_train_sc, y_train,
                         cv=5, scoring='neg_mean_absolute_error')
    results[name] = {
        'MAE':     mean_absolute_error(y_test, preds),
        'RMSE':    np.sqrt(mean_squared_error(y_test, preds)),
        'R2':      r2_score(y_test, preds),
        'CV_mean': -cv.mean(),
        'CV_std':   cv.std(),
        'preds':   preds
    }
    print(f"{name}: MAE={results[name]['MAE']:.4f}  "
          f"RMSE={results[name]['RMSE']:.4f}  "
          f"R2={results[name]['R2']:.4f}  "
          f"CV={results[name]['CV_mean']:.4f}±{results[name]['CV_std']:.4f}")


## 7. Model Evaluation and Analysis
### 7.1 Performance Comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
for i, (metric, label) in enumerate(zip(
        ['MAE','RMSE','R2'],
        ['Mean Absolute Error (USD)','Root Mean Squared Error (USD)','R² Score'])):
    vals = [results[m][metric] for m in results]
    bars = axes[i].bar(list(results.keys()), vals, color=colors, edgecolor='white')
    axes[i].set_title(label, fontweight='bold', fontsize=11)
    axes[i].set_xticklabels(list(results.keys()), rotation=15, ha='right', fontsize=8)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                     f'{v:.2f}' if metric != 'R2' else f'{v:.4f}',
                     ha='center', fontsize=8)
plt.suptitle('Model Performance Comparison', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 7.2 Actual vs Predicted Prices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, mname in zip(axes, ['Linear Regression','Random Forest','Gradient Boosting']):
    preds = results[mname]['preds']
    ax.scatter(y_test, preds, alpha=0.25, s=10, color='steelblue')
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
    ax.set_title(f'{mname}\nMAE=${results[mname]["MAE"]:.2f}, R²={results[mname]["R2"]:.4f}',
                 fontweight='bold', fontsize=9)
    ax.set_xlabel('Actual Amount (USD)'); ax.set_ylabel('Predicted Amount (USD)')
    ax.legend(fontsize=8)
plt.suptitle('Actual vs Predicted Purchase Amounts', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


### 7.3 Cross-Validation Results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cv_means = [results[m]['CV_mean'] for m in results]
cv_stds  = [results[m]['CV_std']  for m in results]
ax.bar(np.arange(4), cv_means, yerr=cv_stds, capsize=6,
       color=colors, alpha=0.85, edgecolor='black')
ax.set_xticks(np.arange(4))
ax.set_xticklabels(list(results.keys()), fontsize=9)
ax.set_title('5-Fold Cross-Validation MAE', fontweight='bold', fontsize=13)
ax.set_ylabel('Mean Absolute Error (USD)')
plt.tight_layout(); plt.show()


### 7.4 Feature Coefficients — Linear Regression

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
coef = pd.Series(lr.coef_, index=selected_features).sort_values()
coef.plot(kind='barh',
          color=['#DD8452' if c > 0 else '#4C72B0' for c in coef], ax=ax)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Linear Regression Feature Coefficients', fontweight='bold', fontsize=13)
ax.set_xlabel('Coefficient Value')
plt.tight_layout(); plt.show()


## 8. Global and Local Explanations with SHAP
### 8.1 SHAP Feature Importance (Global)


In [ ]:
explainer  = shap.TreeExplainer(gbr)
shap_values = explainer.shap_values(X_test_sc)

shap.summary_plot(shap_values, X_test_sc,
                  feature_names=selected_features, plot_type='bar')


### 8.2 SHAP Beeswarm Plot

In [ ]:
shap.summary_plot(shap_values, X_test_sc,
                  feature_names=selected_features)


## 9. Partial Dependency Plots (PDP)


In [ ]:
shap_imp = pd.Series(np.abs(shap_values).mean(axis=0),
                     index=selected_features).sort_values(ascending=False)
top2 = shap_imp.index[:2].tolist()
top2_idx = [selected_features.index(f) for f in top2]
print("Top 2 features for PDP:", top2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
PartialDependenceDisplay.from_estimator(rfr, X_test_sc,
    [top2_idx[0]], feature_names=selected_features, ax=axes[0])
axes[0].set_title(f'PDP: {top2[0]}', fontweight='bold')

PartialDependenceDisplay.from_estimator(rfr, X_test_sc,
    [top2_idx[1]], feature_names=selected_features, ax=axes[1])
axes[1].set_title(f'PDP: {top2[1]}', fontweight='bold')
plt.suptitle('Partial Dependency Plots (Random Forest)', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()


## 10. Conclusion
The analysis reveals that purchase amount in this dataset is essentially uniformly distributed between \$20 and \$100, resulting in near-zero R² scores across all models. This is a meaningful finding: **no combination of demographic, behavioural, or transactional features reliably predicts how much a customer spends per transaction**. The key drivers of shopping behaviour identified through EDA and SHAP are seasonal variation, location, discount/promo usage, and shipping preference — but these influence *what* customers buy more than *how much* they spend.
